In [ ]:
%load_ext autoreload
%autoreload 2
%xmode verbose

In [ ]:
from ExoRM import read_rm_data, load_model, ForecasterRM, preprocess_data

import numpy
import matplotlib.pyplot as plot
import matplotlib
import pandas
import seaborn
import math
import time

plot.style.use('seaborn-v0_8')
matplotlib.rcParams['figure.figsize'] = [8, 6]
matplotlib.rcParams['axes.labelsize'] = 14  # Axis label font size
matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['figure.dpi'] = 300

data = read_rm_data()
data = preprocess_data(data)
data

In [ ]:
columns = ['radius', 'mass']

x = data['radius']
y = data['mass']

x = numpy.log10(x)
y = numpy.log10(y)
seaborn.kdeplot(numpy.log10(data[columns]), x = 'radius', y = 'mass', fill = False, cmap = 'magma', levels = 10, zorder = 1)
# seaborn.scatterplot(numpy.log10(data[columns]), x = 'radius', y = 'mass', s = 5, color = 'black', zorder = 2)
# plot.gca().set_aspect('auto')
plot.xlim(-0.5, 1.75)
plot.ylim(-1, 5)

plot.xlabel('Log Radius (Earth Radii)')
plot.ylabel('Log Mass (Earth Mass)')

In [ ]:
model = load_model()

xs = numpy.linspace(x.min(), x.max(), 10000)

ms = model(xs)

ms2 = ForecasterRM.forecaster(xs)

plot.scatter(x, y, s = 2)

plot.plot(xs, ms, color = 'C1')
plot.plot(xs, ms2, '--', color = 'C2')

plot.fill_between(xs, ms - model.error, ms + model.error, color = 'C1', alpha = 0.1)

plot.legend(['target', 'ExoRM', 'forecaster'])

plot.xlabel('Log Radius (Earth Radii)')
plot.ylabel('Log Mass (Earth Mass)')

plot.show()

In [ ]:
plot.scatter(x, numpy.log10((10 ** y) / ((10 ** x) ** 3)), s = 2)
plot.plot(xs, numpy.log10((10 ** ms) / ((10 ** xs) ** 3)), color = 'C1')
plot.plot(xs, numpy.log10((10 ** ms2) / ((10 ** xs) ** 3)), '--', color = 'C2')

plot.fill_between(xs,
                  numpy.log10((10 ** (ms - model.error)) / ((10 ** xs) ** 3)),
                  numpy.log10((10 ** (ms + model.error)) / ((10 ** xs) ** 3)), color = 'C1', alpha = 0.1)

plot.legend(['target', 'model', 'forecaster'])
plot.xlabel('Log Radius (Earth Radii)')
plot.ylabel('Log Density (g / cm^3)')

plot.show()

In [ ]:
ms1ts = []
ms2ts = []

for i in range(100):
    start_time = time.time()
    model(xs)
    ms1t = time.time() - start_time

    start_time = time.time()
    ForecasterRM.forecaster(xs)
    ms2t = time.time() - start_time

    ms1ts.append(ms1t)
    ms2ts.append(ms2t)

ms1t = numpy.mean(ms1ts) * 1e3
ms2t = numpy.mean(ms2ts) * 1e3
print('ExoRM time (ms): ', ms1t)
print('Forecaster time (ms): ', ms2t)

In [ ]:
p_data = data
columns = ['radius', 'mass']
p_data[columns] = numpy.log10(p_data[columns])
p_data

In [ ]:
p_data['ExoRM'] = model(p_data['radius'])
p_data['Forecaster'] = ForecasterRM.forecaster(p_data['radius'])

p_data['ExoRM res'] = (p_data['mass'] - p_data['ExoRM']).abs()
p_data['Forecaster res'] = (p_data['mass'] - p_data['Forecaster']).abs()

p_data = p_data.sort_values(by = 'name', key = lambda x: x.str.len()).reset_index(drop = True)

columns = ['radius', 'mass', 'ExoRM', 'Forecaster', 'ExoRM res', 'Forecaster res']

p_data[columns] = p_data[columns].map(
    lambda x: x if x == 0 or math.isnan(x) else round(x, (5 - 1) - int(math.floor(math.log10(abs(x)))))
)

p_data[['name'] + columns].to_csv('ExoRM_results.csv', index = False)
p_data.head(10)

In [ ]:
p_data_long = pandas.melt(
    p_data,
    value_vars = ['ExoRM res', 'Forecaster res'],
    var_name = 'Model',
    value_name = 'Residual'
)

p_data_long['Model'] = p_data_long['Model'].map(lambda x: 'ExoRM' if x == 'ExoRM res' else 'Forecaster')
seaborn.violinplot(data = p_data_long, x = 'Model', y = 'Residual', hue = 'Model', palette = 'viridis', zorder = 1, bw_method = 0.3, cut = 0)

plot.title('Log Residuals by Model')
plot.show()
p_data_long